# Interest-Rate Swap Valuation

## Objective

This notebook examines whether the method used to interpolate the yield curve
affects the valuation of a plain-vanilla fixed-for-floating interest-rate swap.

The same swap is valued using:

1. a term structure constructed by interpolating zero rates;
2. a term structure constructed by interpolating discount factors.

The analysis compares par swap rates, off-market swap values and first-order
interest-rate sensitivity under the two curve constructions.

In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

In [2]:
market_data = pd.DataFrame(
    {
        "maturity": [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0],
        "zero_rate": [0.0420, 0.0435, 0.0445, 0.0438, 0.0415, 0.0400, 0.0390],
    }
)

market_data["discount_factor"] = np.exp(
    -market_data["zero_rate"] * market_data["maturity"]
)

curve_maturities = np.linspace(
    market_data["maturity"].min(),
    market_data["maturity"].max(),
    500,
)

zero_rate_interpolator = interp1d(
    market_data["maturity"],
    market_data["zero_rate"],
    kind="linear",
)

discount_factor_interpolator = interp1d(
    market_data["maturity"],
    market_data["discount_factor"],
    kind="linear",
)

zero_rates_from_zero_interpolation = zero_rate_interpolator(
    curve_maturities
)

discount_factors_from_zero_interpolation = np.exp(
    -zero_rates_from_zero_interpolation * curve_maturities
)

discount_factors_from_discount_interpolation = (
    discount_factor_interpolator(curve_maturities)
)

zero_rates_from_discount_interpolation = (
    -np.log(discount_factors_from_discount_interpolation)
    / curve_maturities
)

## Swap specification

Consider a plain-vanilla fixed-for-floating interest-rate swap with notional
\(N\), maturity \(T\), and equally spaced payment dates

$$
0 < t_1 < \cdots < t_n = T.
$$

The fixed leg pays a constant rate \(K\), while the floating leg resets at each
payment date.

Under a single-curve framework, the par swap rate is the fixed rate that makes
the initial value of the swap equal to zero.

In [3]:
notional = 1_000_000
maturity = 5.0
payments_per_year = 2
accrual = 1 / payments_per_year

payment_times = np.arange(
    accrual,
    maturity + accrual,
    accrual,
)

In [4]:
def discount_at_times(
    payment_times: np.ndarray,
    curve_times: np.ndarray,
    discount_factors: np.ndarray,
) -> np.ndarray:
    """
    Interpolate discount factors at the swap payment dates.
    """

    return np.interp(
        payment_times,
        curve_times,
        discount_factors,
    )

In [5]:
discounts_zero_method = discount_at_times(
    payment_times,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

discounts_discount_method = discount_at_times(
    payment_times,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

## Par swap rate

In a single-curve setting, the present value of the floating leg of a spot-starting
swap is

$$
PV_{\mathrm{float}}
=
N\left(1-P(0,T)\right).
$$

The present value of the fixed leg is

$$
PV_{\mathrm{fixed}}
=
NK
\sum_{i=1}^{n}
\alpha_i P(0,t_i),
$$

where \(\alpha_i\) is the accrual fraction for payment period \(i\).

Setting the initial swap value equal to zero gives the par swap rate

$$
K_{\mathrm{par}}
=
\frac{1-P(0,T)}
{\sum_{i=1}^{n}\alpha_i P(0,t_i)}.
$$

In [6]:
def par_swap_rate(
    payment_times: np.ndarray,
    accrual: float,
    curve_times: np.ndarray,
    discount_factors: np.ndarray,
) -> float:
    """
    Return the par fixed rate of a spot-starting vanilla swap.
    """

    payment_discounts = np.interp(
        payment_times,
        curve_times,
        discount_factors,
    )

    annuity = accrual * np.sum(payment_discounts)

    return float(
        (1 - payment_discounts[-1]) / annuity
    )

In [7]:
par_rate_zero = par_swap_rate(
    payment_times,
    accrual,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

par_rate_discount = par_swap_rate(
    payment_times,
    accrual,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

print(
    f"Par swap rate (zero-rate interpolation): "
    f"{100 * par_rate_zero:.6f}%"
)

print(
    f"Par swap rate (discount-factor interpolation): "
    f"{100 * par_rate_discount:.6f}%"
)

print(
    "Difference:",
    f"{10_000 * (par_rate_discount - par_rate_zero):.4f} bp",
)

Par swap rate (zero-rate interpolation): 4.209732%
Par swap rate (discount-factor interpolation): 4.207141%
Difference: -0.2591 bp


## Interpretation

The two interpolation methods produce very similar par swap rates, reflecting
the fact that both curves reproduce the same market observations at the quoted
maturities.

The remaining difference arises solely from the interpolation of discount
factors between those maturities. The next section examines whether this small
difference leads to a more noticeable effect when valuing an off-market swap.

## Off-market swap valuation

A newly initiated swap with fixed rate equal to the par swap rate has zero
value. In practice, however, swaps are frequently valued after market rates
have changed or when the contractual fixed rate differs from the current par
rate.

This section values an off-market payer swap under each interpolated term
structure and compares the resulting valuations.

For a payer swap, the value is

$$
V_{\mathrm{payer}}
=
PV_{\mathrm{float}}
-
PV_{\mathrm{fixed}},
$$

where

$$
PV_{\mathrm{float}}
=
N\left(1-P(0,T)\right),
$$

and

$$
PV_{\mathrm{fixed}}
=
NK
\sum_{i=1}^{n}
\alpha_i P(0,t_i).
$$

In [8]:
def swap_value(
    fixed_rate: float,
    notional: float,
    payment_times: np.ndarray,
    accrual: float,
    curve_times: np.ndarray,
    discount_factors: np.ndarray,
) -> float:
    """
    Return the value of a payer swap under a single-curve framework.
    """

    payment_discounts = np.interp(
        payment_times,
        curve_times,
        discount_factors,
    )

    fixed_leg = (
        notional
        * fixed_rate
        * accrual
        * np.sum(payment_discounts)
    )

    floating_leg = (
        notional
        * (1 - payment_discounts[-1])
    )

    return floating_leg - fixed_leg

In [9]:
fixed_rate = 0.045

In [10]:
swap_value_zero = swap_value(
    fixed_rate,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

swap_value_discount = swap_value(
    fixed_rate,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

print(
    f"Swap value (zero-rate interpolation): {swap_value_zero:,.2f}"
)

print(
    f"Swap value (discount-factor interpolation): {swap_value_discount:,.2f}"
)

print(
    "Difference:",
    f"{swap_value_discount - swap_value_zero:,.2f}"
)

Swap value (zero-rate interpolation): -12,921.13
Swap value (discount-factor interpolation): -13,042.85
Difference: -121.72


## Interpretation

The 4.5% contractual fixed rate lies above the par rate implied by either
curve, so the payer swap has negative value under both term structures.

The interpolation choice changes the implied par rate by approximately
0.26 basis points and produces a valuation difference of approximately
122 on a notional of 1 million.

Although this difference is small relative to the notional, it is measurable
and arises solely from the construction of the term structure. This illustrates
how differences that appear minor at the curve level can propagate into the
valuation of an interest-rate instrument.

## Validation

As a consistency check, a swap struck at the par rate implied by a given
term structure should have approximately zero value when valued using that
same term structure.

In [11]:
par_value_zero = swap_value(
    par_rate_zero,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

par_value_discount = swap_value(
    par_rate_discount,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

assert np.isclose(par_value_zero, 0.0, atol=1e-8)
assert np.isclose(par_value_discount, 0.0, atol=1e-8)

print("Par swap valuation checks passed.")

Par swap valuation checks passed.


## Parallel curve shocks and DV01

To measure first-order interest-rate sensitivity, each zero-rate curve is
shifted in parallel by \( \pm 1 \) basis point and the off-market payer swap
is repriced.

Using a central finite-difference approximation,

$$
\mathrm{DV01}
=
\frac{V_{-1\mathrm{bp}} - V_{+1\mathrm{bp}}}{2},
$$

where \(V_{\pm1\mathrm{bp}}\) denotes the swap value after a parallel
\(\pm1\) basis-point shift in the zero curve.

In [12]:
def shock_zero_rates(
    zero_rates: np.ndarray,
    shock_bp: float,
) -> np.ndarray:
    """Apply a parallel basis-point shock to a zero-rate curve."""

    return zero_rates + shock_bp / 10_000


def zero_rates_to_discount_factors(
    zero_rates: np.ndarray,
    maturities: np.ndarray,
) -> np.ndarray:
    """Convert continuously compounded zero rates to discount factors."""

    return np.exp(-zero_rates * maturities)

In [13]:
zero_curve_up = shock_zero_rates(
    zero_rates_from_zero_interpolation,
    1,
)

zero_curve_down = shock_zero_rates(
    zero_rates_from_zero_interpolation,
    -1,
)

discount_zero_up = zero_rates_to_discount_factors(
    zero_curve_up,
    curve_maturities,
)

discount_zero_down = zero_rates_to_discount_factors(
    zero_curve_down,
    curve_maturities,
)

In [14]:
discount_curve_up = shock_zero_rates(
    zero_rates_from_discount_interpolation,
    1,
)

discount_curve_down = shock_zero_rates(
    zero_rates_from_discount_interpolation,
    -1,
)

discount_discount_up = zero_rates_to_discount_factors(
    discount_curve_up,
    curve_maturities,
)

discount_discount_down = zero_rates_to_discount_factors(
    discount_curve_down,
    curve_maturities,
)

In [15]:
swap_zero_up = swap_value(
    fixed_rate,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_zero_up,
)

swap_zero_down = swap_value(
    fixed_rate,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_zero_down,
)

swap_discount_up = swap_value(
    fixed_rate,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_discount_up,
)

swap_discount_down = swap_value(
    fixed_rate,
    notional,
    payment_times,
    accrual,
    curve_maturities,
    discount_discount_down,
)

In [16]:
print("Zero-rate interpolation")
print(f"Value (+1 bp): {swap_zero_up:,.2f}")
print(f"Value (-1 bp): {swap_zero_down:,.2f}")

print()

print("Discount-factor interpolation")
print(f"Value (+1 bp): {swap_discount_up:,.2f}")
print(f"Value (-1 bp): {swap_discount_down:,.2f}")

Zero-rate interpolation
Value (+1 bp): -12,461.57
Value (-1 bp): -13,380.91

Discount-factor interpolation
Value (+1 bp): -12,583.24
Value (-1 bp): -13,502.68


In [17]:
def dv01(
    value_down: float,
    value_up: float,
) -> float:
    """Estimate DV01 using a central finite-difference approximation."""

    return (value_down - value_up) / 2

In [18]:
dv01_zero = dv01(
    swap_zero_down,
    swap_zero_up,
)

dv01_discount = dv01(
    swap_discount_down,
    swap_discount_up,
)

print(
    f"DV01 (zero-rate interpolation): {dv01_zero:,.4f}"
)

print(
    f"DV01 (discount-factor interpolation): {dv01_discount:,.4f}"
)

print(
    "Difference:",
    f"{dv01_discount - dv01_zero:,.6f}",
)

DV01 (zero-rate interpolation): -459.6691
DV01 (discount-factor interpolation): -459.7189
Difference: -0.049764


## Interpretation

The payer swap has a DV01 of approximately \(-460\) under both curve
constructions. The negative sign reflects the direction of the position:
a parallel increase in rates increases the value of the payer swap, while a
decrease in rates reduces its value.

The difference between the two DV01 estimates is only about \(0.05\), despite
the approximately \(122\) difference in the unshocked swap valuations.

For this example, the interpolation method therefore has a measurable effect
on the level of the swap valuation but very little effect on its sensitivity
to a small parallel shift in the yield curve.

## Model assumptions and limitations

The analysis uses a deliberately simplified single-curve framework in order to
isolate the effect of interpolation.

In particular:

- the same term structure is used for discounting and the floating leg;
- rates are represented by continuously compounded zero rates;
- curve shifts are assumed to be parallel;
- counterparty credit risk, collateral and funding effects are ignored;
- payment dates are equally spaced and a constant accrual fraction is used.

In modern interest-rate markets, discounting and forward-rate projection are
typically treated separately. Extending the analysis to a multi-curve
framework would therefore be a natural next step, but is outside the scope of
this project.

# Findings

This notebook investigated whether yield-curve interpolation affects the
valuation and first-order interest-rate risk of a vanilla interest-rate swap.

The two curve constructions produced par swap rates differing by approximately
0.26 basis points. For a 4.5% payer swap with notional 1 million, this translated
into a valuation difference of approximately 122.

In contrast, the corresponding DV01 estimates were almost identical, differing
by only about 0.05. For the instrument considered here, the interpolation
choice therefore affected the level of valuation more noticeably than its
sensitivity to a small parallel movement in rates.

Together with the bond analysis, these results show how differences introduced
during curve construction can propagate into downstream valuation, while their
effect on standard parallel-rate risk measures may remain comparatively small.